# Run everything
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Syndr0/compute-efficient-genqa/blob/main/notebooks/run_all.ipynb)

Runs every stage in order (data → baselines → RQ1 → RQ2 → RQ3 → tiny Transformer → report).
Tick `SMOKE_TEST` for a ~10-minute end-to-end check on a handful of examples.
Full run: about 2–3 h on a T4, or 1.5–2 h on an A100 with the larger A100 profile
(`configs/a100.yaml`, picked automatically for any GPU with ≥ 40 GB: A100, H100, G4). Every stage resumes from what is already on Drive,
so after a disconnect just run the cells again.

In [ ]:
#@title Setup: clone the repo, install, choose where outputs go
#@markdown Outputs (data, adapters, predictions, metrics) persist on Google Drive so a
#@markdown disconnected runtime can resume: every stage skips work that is already saved.
USE_DRIVE = True  #@param {type:"boolean"}
SMOKE_TEST = False  #@param {type:"boolean"}
GPU_PROFILE = "auto"  #@param ["auto", "t4", "a100"]
import os, sys
REPO_DIR = "/content/compute-efficient-genqa"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/Syndr0/compute-efficient-genqa.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!pip install -q -e .
sys.path.insert(0, f"{REPO_DIR}/src")
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/cegqa_outputs"
else:
    OUT = "/content/cegqa_outputs"
import torch
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
GPU_MEM_GB = torch.cuda.get_device_properties(0).total_memory / 2**30 if torch.cuda.is_available() else 0
if GPU_PROFILE == "auto":  # large-GPU profile for A100 / H100 / G4 (RTX PRO 6000) and anything with >= 40 GB
    GPU_PROFILE = "a100" if GPU_MEM_GB >= 39 else "t4"
CONFIG_FILES = ["configs/default.yaml"]
if GPU_PROFILE == "a100":
    CONFIG_FILES.append("configs/a100.yaml")
    OUT += "_a100"  # different budgets/sizes: keep results of the two profiles apart
if SMOKE_TEST:
    CONFIG_FILES.append("configs/smoke.yaml")
    OUT += "_smoke"
os.environ["CEGQA_OUTPUT_DIR"] = OUT
CONFIG = " ".join(f"--config {f}" for f in CONFIG_FILES)
print(f"GPU: {GPU} ({GPU_MEM_GB:.0f} GB) | profile: {GPU_PROFILE} | outputs -> {OUT}")
print("configs:", CONFIG_FILES)

In [ ]:
!python -m cegqa.cli all $CONFIG

In [ ]:
from IPython.display import Markdown, display
import os
os.chdir(OUT)
display(Markdown(open("report.md").read()))
os.chdir(REPO_DIR)